# 02 — Silver Transform

**Layer:** Silver · **Day:** 1 · **Authoritative script:** `scripts/day1_build_lakehouse.py` (function `build_silver`)

## Objective

Turn the bronze raw table into a cleaned, typed, and anonymised employee table that is safe to use for analytics and model training.

## Inputs

- `lakehouse/bronze/employees_raw.parquet`

## Outputs

- `lakehouse/silver/employees.parquet`
  - **Anonymised `EmployeeID`** — SHA-256 of `EmployeeNumber` salted with the project key (`agile-hr-copilot-2026`), truncated to 10 chars (e.g. `EMP_3F9A21BC74`). The original `EmployeeNumber` is dropped.
  - **Bucketed demographic-adjacent fields** for k-anonymity-friendly reporting:
    - `AgeBand` — `18-24`, `25-34`, `35-44`, `45-54`, `55+`
    - `SalaryBand` — Bands 1 to 5 by monthly income
    - `TenureCohort` — `0-1y (new)`, `1-3y`, `3-5y`, `5-10y`, `10y+`
  - **Renames** — `EnvSatisfaction`, `PriorCompanies`, `LastRaisePct`, `RelationSatisfaction`, `TotalExperienceYears`, `TrainingCount`, `TenureYears`, `YearsInRole`, `YearsSincePromotion`, `YearsWithManager` — for readability and consistency with the Power BI semantic model.
  - **Dropped constants** — `EmployeeCount`, `Over18`, `StandardHours`.
  - **`AttritionFlag`** — integer target (0/1) derived from the `Attrition` Yes/No column.

## Business value

Silver is the layer everyone (analytics, Power BI, ML) actually consumes. Cleaning here is done **once** and the contract is documented, so the model and the dashboards are always looking at the same world.

## Reproduce

```powershell
python scripts\day1_build_lakehouse.py
```

In [ ]:
from pathlib import Path
import pandas as pd

SILVER = Path('../lakehouse/silver/employees.parquet')
print('exists:', SILVER.exists())

In [ ]:
if SILVER.exists():
    df = pd.read_parquet(SILVER)
    print(f'rows: {len(df):,}  cols: {df.shape[1]}')
    print('attrition rate:', round(df["AttritionFlag"].mean(), 3))
    print('\nbanded fields preview:')
    df[['EmployeeID','AgeBand','SalaryBand','TenureCohort','AttritionFlag']].head(5)

## Privacy notes

- `EmployeeID` is a stable but **non-reversible** label. The salt lives in code so the mapping cannot be reproduced from the parquet alone.
- Bucketed fields make group-level reporting safer — no single demographic value identifies an individual.
- Power BI dashboards layer a k-anonymity threshold on top (cohorts < 25 are suppressed).

## Interview talking points

- Silver-vs-bronze separation lets us replay the cleaning step independently of ingest.
- The dropped constants are a small but real example of *governance through data shape* — they cannot be misused if they don't exist downstream.
- All renames are conservative (semantic, not behavioural) so existing notebooks and SQL still work after a refactor.